In [ ]:
import argparse
import sys
from pathlib import Path
import os
import pandas as pd

# In a notebook, abspath('') is already this notebook's directory
# (architecture/preprocessing) — no .parent, unlike __file__ in the script.
_HERE = Path(os.path.abspath('')).resolve()
ARCH_DIR = _HERE.parent
REPO_ROOT = ARCH_DIR.parent
# fed_stroke (frozen schema) + repo root (the shared `preprocessing` library).
for _p in (ARCH_DIR, REPO_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


from fed_stroke.schema import FEATURE_COLS, FEATURE_UNITS, TARGET_COL  # noqa: E402
from preprocessing.case_ids import (
    create_ehr_case_identification_column,
    create_registry_case_identification_column,
)
from preprocessing.registry_cohort import (
    build_cohort,
    parse_yyyymmdd,
    preprocess_features,
    preprocess_outcome,
)

SCHEMA_VERSION = "frozen-v1"   # bump on any FEATURE_COLS/FEATURE_UNITS/TARGET_COL change
PROVENANCE = "real-frozen-schema"


def build_frozen_gva_table(registry_xlsx: Path, ehr_dir: Path) -> pd.DataFrame:
    """Registry + EHR → ONE tidy per-admission table in the frozen schema.

    Steps to populate (reuse, don't re-derive):
    - cohort: preprocessing.registry_cohort (exact-duplicate rows dropped,
      'Type of event' == 'Ischemic stroke');
    - outcome: the OPSUM 3M Death / 3M mRS reconciliation already coded there;
    - case_admission_id: preprocessing.case_ids
      (== the loader's patient_id + '_' + EDS-last-4 derivation, task.py);
    - EHR features: preprocessing.first_values extraction, joined on
      case_admission_id;
    - unit-convert into FEATURE_UNITS (preprocessing.mappings.UNIT_CONVERSIONS),
      rename to the frozen names (preprocessing.mappings.GVA_TO_FROZEN), then
      preprocessing.mappings.validate_frozen_columns.

    Input:
        registry_xlsx: the Geneva stroke-registry export (.xlsx).
        ehr_dir: the EHR extraction directory (patientvalue + lab CSVs).
    Output:
        DataFrame, one row per case_admission_id, columns EXACTLY
        ['case_admission_id', *FEATURE_COLS, TARGET_COL]:
        - case_admission_id: str, '<patient_id>_<eds_last_4>';
        - features: float, IN FEATURE_UNITS, missing = NaN (no sentinel),
          out-of-range values resolved HERE (data-quality error, not a
          privacy question);
        - target: int in {0, 1}; rows with underivable outcome dropped
          (count reported via the summary).
    """
    # preprocess registry
    df = pd.read_excel(registry_xlsx)
    df, n_raw, n_filtered = build_cohort(df)
    # derive case_admission_id
    df['case_admission_id'] = create_registry_case_identification_column(df)
    df = preprocess_outcome(df)
    df = preprocess_features(df)

    return df


In [ ]:
registry_xlsx_path = '/Users/jk/stroke_datasets/stroke_registry_post_hoc_modified.xlsx'
ehr_dir = '/Users/jk/stroke_datasets/Extraction_20220815'

In [ ]:
df = build_frozen_gva_table(Path(registry_xlsx_path), Path(ehr_dir))

In [ ]:
df.shape

In [ ]:
df["admission_date"] = parse_yyyymmdd(df["Arrival at hospital"])

In [ ]:
from preprocessing.first_values import (  # noqa: E402
    assemble_wide,
    extract_lab_dosage_first_values,
    extract_pv_lab_first_values,
    extract_pv_vital_first_values,
    load_concat_csvs,
)
ehr_dir = Path(ehr_dir)

vitals_prefix = "patientvalue"
lab_prefix = "lab"
print(f"[df] n_patients={len(df)}")

print(f"[load]   PV files ({vitals_prefix}*.csv) from {ehr_dir}")
vitals_df = load_concat_csvs(ehr_dir, vitals_prefix)
vitals_df["case_admission_id"] = create_ehr_case_identification_column(vitals_df)
print(f"[load]   PV rows={len(vitals_df)}")

print(f"[load]   lab files ({lab_prefix}*.csv) from {ehr_dir}")
lab_df = load_concat_csvs(ehr_dir, lab_prefix)
lab_df["case_admission_id"] = create_ehr_case_identification_column(lab_df)
print(f"[load]   lab rows={len(lab_df)}")

per_var: dict[str, pd.DataFrame] = {}
per_var.update(extract_pv_vital_first_values(vitals_df, df))
per_var.update(extract_pv_lab_first_values(vitals_df, df))
per_var.update(extract_lab_dosage_first_values(lab_df, df))

wide_df = assemble_wide(df, per_var)


In [ ]:
wide_df['Time of symptom onset known'].unique()

In [ ]:
wide_df